# RedZone EXP-1404: re-grade the commentary importance on REAL outcomes (CPU, Data planner E)

EXP-1402's basketball importance AUC (NBA 0.955, CBB 0.957) used `big_next`, which for basketball is *a field goal in the
last 2:00 of the 4th (CBB: 2nd half) within 5*: our own clock rule, which the Teacher also sees. This notebook re-labels
the **same 66,048 answered plays** with lane 3's outcome label (lead change or a 20-point win-probability swing in the
next few minutes, `redzone/lab/outcome_label.py`) and writes `exp1404_labels.zip`. Hand that zip to Data planner E,
who joins it to the private Teacher answers and grades (nothing private is in this notebook: game ids and play numbers only).

**CPU runtime is enough** (Colab: Runtime > Change runtime type > CPU; or a Kaggle CPU notebook with Internet on).
Then Run all. ~2,500 ESPN game summaries (public site API, plan B personal use), ~15-30 min, ~0 CU.
Code: public commit `c6a28b4` of this repo.

In [ ]:
PIN = "c6a28b4d0cb2794be6e0642f42ac5fa92c8c90ff"
REPO = "BouncySlime1215/redzone-everything"
DATA_PLAN = "B"   # ESPN, personal use only

In [ ]:
import os, subprocess, sys, tarfile, time, urllib.request, zipfile
RUN = "/kaggle/working/rz1404" if os.path.isdir("/kaggle/working") else "/content/rz1404"
CODE = f"{RUN}/code"
os.makedirs(RUN, exist_ok=True)
if not os.path.exists(f"{CODE}/.done"):
    for i in range(4):
        try:
            urllib.request.urlretrieve(f"https://codeload.github.com/{REPO}/tar.gz/{PIN}", f"{RUN}/x.tar.gz"); break
        except Exception as e:
            print("retry", e); time.sleep(20 * (i + 1))
    with tarfile.open(f"{RUN}/x.tar.gz") as t:
        for m in t.getmembers():
            parts = m.name.split("/", 1)
            if len(parts) == 2 and parts[1] and ".." not in parts[1].split("/"):
                m.name = parts[1]; t.extract(m, CODE, filter="data")
    os.remove(f"{RUN}/x.tar.gz"); open(f"{CODE}/.done", "w").close()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx", "pyyaml"], check=False)
import gzip, shutil
with gzip.open(f"{CODE}/agent/kaggle/assets/exp1404_games.json.gz") as f, open(f"{RUN}/games.json", "wb") as g:
    shutil.copyfileobj(f, g)
print("code ready")

In [ ]:
t0 = time.time()
env = {**os.environ, "PYTHONPATH": CODE, "PYTHONUNBUFFERED": "1", "RZ_DATA_PLAN": DATA_PLAN}
subprocess.run([sys.executable, "-u", "agent/kaggle/commentary_regrade.py", "label", "--games", f"{RUN}/games.json",
                "--out", f"{RUN}/exp1404_labels.jsonl.gz"], cwd=CODE, env=env, check=True)
with zipfile.ZipFile(f"{RUN}/exp1404_labels.zip", "w") as z:
    z.write(f"{RUN}/exp1404_labels.jsonl.gz", "exp1404_labels.jsonl.gz")
    z.writestr("README.txt", f"EXP-1404 outcome labels for the EXP-1402 answered plays. Code {PIN}. "
               "ESPN-derived, personal-use, not for commercial release. Do not publish.\n")
print(f"done in {(time.time()-t0)/60:.1f} min ->", f"{RUN}/exp1404_labels.zip")
try:
    from google.colab import files; files.download(f"{RUN}/exp1404_labels.zip")
except ImportError:
    print("Kaggle: download exp1404_labels.zip from the Output tab")